# LeetCode 1858 — Longest Word With All Prefixes bằng Python

Link đề: https://leetcode.com/problems/longest-word-with-all-prefixes/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Trie, Hash Table, Sorting

> **Lưu ý: đây là bài Premium** trên LeetCode — cần tài khoản Premium để xem đề gốc và nộp bài. Đề dưới đây viết theo hiểu biết về bài (nguồn: bản đề của doocs/leetcode); nếu có tài khoản Premium, hãy **đối chiếu lại đề gốc**.

Skeleton LeetCode:

```python
class Solution:
    def longestWord(self, words: List[str]) -> str:
        ...
```

## 1. Mục tiêu học

1. Hiểu điều kiện "**mọi tiền tố** của từ đều có trong danh sách" và cách kiểm tra nó hiệu quả.
2. Luyện **Trie** (cây tiền tố): chèn, đánh dấu kết thúc từ, đi theo đường và kiểm tra cờ ở **mỗi** nút.
3. Thấy mẹo **sắp xếp**: sau khi sắp xếp, tiền tố luôn đứng trước từ dài hơn ⇒ kiểm tra "tiền tố bỏ một ký tự đã hợp lệ chưa" (quy hoạch động trên tập hợp).
4. Xử lý **tie-break**: cùng độ dài thì chọn từ **nhỏ nhất theo thứ tự từ điển**; không có thì trả `""`.
5. Phân tích độ phức tạp theo **tổng độ dài** `T = Σ|words[i]|` thay vì số từ.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `words`. Tìm chuỗi **dài nhất** trong `words` sao cho **mọi tiền tố** của nó cũng có trong `words`.

Ví dụ: với `words = ["a", "app", "ap"]`, chuỗi `"app"` có các tiền tố `"ap"` và `"a"` đều nằm trong `words`.

Trả về chuỗi tìm được. Nếu có nhiều chuỗi cùng độ dài lớn nhất, trả chuỗi **nhỏ nhất theo thứ tự từ điển**; nếu không có chuỗi nào thỏa, trả `""`.

**Ví dụ 1:** `words = ["k","ki","kir","kira","kiran"]` → `"kiran"`.

**Ví dụ 2:** `words = ["a","banana","app","appl","ap","apply","apple"]` → `"apple"` (cả `"apple"` và `"apply"` đều thỏa; `"apple"` nhỏ hơn theo từ điển).

**Ví dụ 3:** `words = ["abc","bc","ab","qwe"]` → `""`.

**Ràng buộc:** `1 ≤ words.length ≤ 10⁵`; `1 ≤ words[i].length ≤ 10⁵`; `1 ≤ Σ words[i].length ≤ 10⁵`; chỉ chữ thường.

*(Đề Premium — đã diễn đạt lại, hãy đối chiếu lại đề gốc.)*

In [1]:
import random
import string
import time
from typing import List

examples = [
    (["k", "ki", "kir", "kira", "kiran"], "kiran"),
    (["a", "banana", "app", "appl", "ap", "apply", "apple"], "apple"),
    (["abc", "bc", "ab", "qwe"], ""),
]
for words, exp in examples:
    print(f"{words} -> kỳ vọng {exp!r}")

['k', 'ki', 'kir', 'kira', 'kiran'] -> kỳ vọng 'kiran'
['a', 'banana', 'app', 'appl', 'ap', 'apply', 'apple'] -> kỳ vọng 'apple'
['abc', 'bc', 'ab', 'qwe'] -> kỳ vọng ''


## 3. Kiến thức nền

### 3.1 Tiền tố của một chuỗi

Tiền tố của `w` là `w[:k]` với `1 ≤ k ≤ len(w)` (đề tính cả chính `w`, nhưng `w` vốn đã nằm trong `words`). Chuỗi dài `m` có `m` tiền tố khác rỗng.

In [2]:
w = "apple"
print([w[:k] for k in range(1, len(w) + 1)])

['a', 'ap', 'app', 'appl', 'apple']


### 3.2 Tập hợp (set) và chi phí băm chuỗi

Tra `x in S` trung bình `O(1)` **lần băm**, nhưng băm một chuỗi dài `m` tốn `O(m)` (Python cache hash của chính đối tượng chuỗi, còn mỗi lát cắt `w[:k]` là chuỗi **mới** nên phải băm lại). Kiểm tra mọi tiền tố của `w` bằng set ⇒ `O(m²)`.

### 3.3 Trie (cây tiền tố)

Mỗi nút là một dict `{ký tự: nút con}` cộng cờ `end` (có từ kết thúc ở đây). Đường từ gốc tới một nút đại diện cho một tiền tố. Chèn từ dài `m`: `O(m)`.

In [3]:
END = "$"
def make_trie(words):
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node[END] = True
    return root

def show_trie(node, prefix="", depth=0, out=None):
    out = [] if out is None else out
    for ch in sorted(k for k in node if k != END):
        child = node[ch]
        out.append("  " * depth + ch + (" (end)" if END in child else ""))
        show_trie(child, prefix + ch, depth + 1, out)
    return out

print("\n".join(show_trie(make_trie(["a", "app", "ap", "b"]))))

a (end)
  p (end)
    p (end)
b (end)


### 3.4 Thứ tự từ điển và sắp xếp chuỗi

Trong Python, `"app" < "apple" < "apply" < "b"`. Một tính chất quan trọng: **tiền tố luôn nhỏ hơn** chuỗi chứa nó (`"ap" < "app"`). Vậy sau `sorted(words)`, mọi tiền tố của `w` (nếu có) đều đứng **trước** `w`.

In [4]:
ws = ["apply", "b", "app", "a", "apple", "ap", "appl"]
print(sorted(ws))
print(sorted(ws, key=lambda x: (-len(x), x))[:3], "<- sắp (độ dài giảm, từ điển tăng)")

['a', 'ap', 'app', 'appl', 'apple', 'apply', 'b']
['apple', 'apply', 'appl'] <- sắp (độ dài giảm, từ điển tăng)


### 3.5 Đo chi phí: cắt chuỗi + băm tăng theo độ dài

Thí nghiệm nhỏ: kiểm tra mọi tiền tố của một từ dài `m` bằng set. Thời gian tăng xấp xỉ **bình phương** theo `m` (gấp đôi `m` ⇒ khoảng gấp bốn thời gian), khẳng định nhận xét ở 3.2.

In [5]:
for m in (2000, 4000, 8000):
    w = "a" * m
    S = {w[:k] for k in range(1, m + 1)}
    t = time.perf_counter()
    ok = all(w[:k] in S for k in range(1, m + 1))
    print(f"m={m:>5}: {ok} ({(time.perf_counter() - t) * 1000:.1f} ms)")

m= 2000: True (0.8 ms)
m= 4000: True (3.0 ms)
m= 8000: True (11.6 ms)


## 4. Ý tưởng nền tảng / trực giác

Gọi từ `w` là **hợp lệ** nếu mọi tiền tố của nó nằm trong `words`. Quan sát then chốt:

> `w` hợp lệ ⇔ `w` có trong `words` **và** (`len(w) = 1` hoặc `w[:-1]` hợp lệ).

Tức là hợp lệ là tính chất **truyền dọc** theo chuỗi tiền tố. Trên Trie, nghĩa là: đi từ gốc tới nút của `w`, **mọi nút trên đường** đều có cờ `end`.

Tính tay **Ví dụ 2**:

| từ | tiền tố | có đủ? |
|---|---|---|
| a | a | ✓ |
| ap | a, ap | ✓ |
| app | a, ap, app | ✓ |
| appl | a, ap, app, appl | ✓ |
| apple | … , apple | ✓ (dài 5) |
| apply | … , apply | ✓ (dài 5) |
| banana | b ✗ | ✗ |

Hai ứng viên dài 5: `"apple" < "apply"` ⇒ đáp án `"apple"`.

In [6]:
words, _ = examples[1]
S = set(words)
for w in sorted(words):
    miss = [w[:k] for k in range(1, len(w) + 1) if w[:k] not in S]
    print(f"{w:<7} thiếu: {miss if miss else 'không'}")

a       thiếu: không
ap      thiếu: không
app     thiếu: không
appl    thiếu: không
apple   thiếu: không
apply   thiếu: không
banana  thiếu: ['b', 'ba', 'ban', 'bana', 'banan']


## 5. Cách 1 — Brute Force / baseline: set + kiểm tra từng tiền tố

Cho mỗi từ, kiểm tra mọi tiền tố bằng set (dừng sớm khi thiếu). Cập nhật đáp án khi dài hơn, hoặc bằng độ dài mà nhỏ hơn theo từ điển.

In [7]:
def longest_bruteforce(words):
    S = set(words)
    best = ""
    for w in words:
        if all(w[:k] in S for k in range(1, len(w))):
            if len(w) > len(best) or (len(w) == len(best) and w < best):
                best = w
    return best

for words, exp in examples:
    got = longest_bruteforce(words); print(repr(got), "kỳ vọng", repr(exp)); assert got == exp

'kiran' kỳ vọng 'kiran'
'apple' kỳ vọng 'apple'
'' kỳ vọng ''


**Độ phức tạp:** mỗi từ dài `m` tốn tới `O(m²)` (cắt + băm tiền tố). Thoạt nhìn tệ nhất `O(T²)`, nhưng nhờ **dừng sớm**, một từ chỉ "đi xa" được khi các tiền tố ngắn thật sự tồn tại, mà chúng chiếm tổng độ dài. Nếu có các tiền tố dài `1..k` thì `k²/2 ≤ T` ⇒ `k ≤ √(2T) ≈ 447`, nên mỗi từ tốn `O(k²) = O(T)` và tổng `O(n·T)` tệ nhất về lý thuyết — vẫn có thể chậm với nhiều từ dài chung một chuỗi tiền tố. Các cách sau đạt `O(T)` hoặc `O(T log n)`.

### Cách 1' — Sắp `(−len, word)` rồi trả từ hợp lệ đầu tiên

Đỡ phải tự viết tie-break: thứ tự sắp đã ưu tiên dài trước, từ điển nhỏ trước.

In [8]:
def longest_sorted_first(words):
    S = set(words)
    for w in sorted(words, key=lambda x: (-len(x), x)):
        if all(w[:k] in S for k in range(1, len(w))):
            return w
    return ""

print([longest_sorted_first(w) for w, _ in examples])

['kiran', 'apple', '']


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (truyền dọc):** `w` hợp lệ ⇔ `w ∈ words` và `w[:-1]` hợp lệ (hoặc `len(w) = 1`). Không cần kiểm tra **mọi** tiền tố của `w`, chỉ cần **một** tiền tố liền trước — nếu đã biết kết quả của nó.

**Insight 2 (thứ tự xử lý):** để `w[:-1]` có kết quả trước `w`, xử lý theo **độ dài tăng** (hoặc `sorted` từ điển — tiền tố luôn đứng trước). Dùng một set `valid` để lưu các từ hợp lệ đã thấy.

**Insight 3 (Trie):** điều kiện "mọi tiền tố có mặt" = "mọi nút trên đường có cờ `end`". DFS từ gốc **chỉ đi vào nút có `end`** sẽ duyệt đúng tập các từ hợp lệ; nút sâu nhất là đáp án. Duyệt con theo `a..z` và chỉ cập nhật khi **sâu hơn hẳn** ⇒ tự động được từ nhỏ nhất theo từ điển.

**Invariant (Cách 2):** khi xử lý `w`, `valid` chứa **đúng** các từ hợp lệ có độ dài `< len(w)` (và một số cùng độ dài, vô hại).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp + set các từ hợp lệ (quy hoạch động trên tiền tố)

In [9]:
def longest_sort_valid(words):
    valid = set()
    best = ""
    for w in sorted(words):                   # tiền tố luôn đứng trước
        if len(w) == 1 or w[:-1] in valid:
            valid.add(w)
            if len(w) > len(best):            # '>' nghiêm ngặt: giữ từ đến trước = nhỏ hơn theo từ điển
                best = w
    return best

print([longest_sort_valid(w) for w, _ in examples])

['kiran', 'apple', '']


Vì duyệt theo thứ tự từ điển tăng, từ nào cùng độ dài mà đến trước thì nhỏ hơn ⇒ chỉ cập nhật khi **dài hơn hẳn**.

**Độ phức tạp Cách 2:** sắp xếp `O(T log n)` (so sánh chuỗi), mỗi từ một lần cắt + băm `O(m)` ⇒ tổng `O(T log n)`. Space `O(T)`.

### 7.2 Cách 3 — Trie + kiểm tra cờ `end` trên đường đi của từng từ

In [10]:
def longest_trie_walk(words):
    root = make_trie(words)
    best = ""
    for w in words:
        node, ok = root, True
        for ch in w:
            node = node[ch]
            if END not in node:              # một tiền tố không phải là từ
                ok = False
                break
        if ok and (len(w) > len(best) or (len(w) == len(best) and w < best)):
            best = w
    return best

print([longest_trie_walk(w) for w, _ in examples])

['kiran', 'apple', '']


**Độ phức tạp Cách 3:** chèn `O(T)`, mỗi từ đi `O(m)` ⇒ `O(T)` (cộng `O(m)` cho so sánh tie-break). Space `O(T)` nút.

### 7.3 Cách 4 — DFS trên Trie chỉ qua các nút có `end`

Một lượt DFS (dùng stack, không đệ quy) bắt đầu từ gốc, chỉ đẩy các con có cờ `end`. Ta cần duyệt con theo thứ tự `a → z` và cập nhật khi **sâu hơn hẳn** — với stack, đẩy con theo thứ tự **ngược** `z → a` để `a` được lấy ra trước.

In [11]:
class Solution:
    def longestWord(self, words: List[str]) -> str:
        root = {}
        for w in words:
            node = root
            for ch in w:
                node = node.setdefault(ch, {})
            node["$"] = True
        best = ""
        stack = [(root, "")]
        while stack:
            node, path = stack.pop()
            if len(path) > len(best):
                best = path
            for ch in sorted((k for k in node if k != "$"), reverse=True):
                child = node[ch]
                if "$" in child:
                    stack.append((child, path + ch))
        return best

for words, exp in examples:
    got = Solution().longestWord(words); print(repr(got), repr(exp)); assert got == exp

'kiran' 'kiran'
'apple' 'apple'
'' ''


Lưu ý về tie-break với DFS bằng stack: các nút được lấy ra theo **thứ tự tiền tự (pre-order) từ điển** (con `a` trước `b`…), nên trong các từ cùng độ dài, từ nhỏ nhất theo từ điển được thăm **trước** ⇒ cập nhật bằng `>` nghiêm ngặt là đúng. Chuỗi `path + ch` tốn `O(độ sâu)` mỗi lần, nhưng chỉ tạo cho nút **hợp lệ**, mà tổng độ sâu các nút hợp lệ `≤ T`.

**Độ phức tạp Cách 4:** `O(T · log σ)` (sắp các con, `σ = 26`) ≈ `O(T)`. Space `O(T)`.

## 8. TRACE từng bước

Trace Cách 2 (sắp xếp + `valid`) trên **Ví dụ 2**:

In [12]:
def trace_sort_valid(words):
    valid, best = set(), ""
    for w in sorted(words):
        cond = len(w) == 1 or w[:-1] in valid
        if cond:
            valid.add(w)
            if len(w) > len(best):
                best = w
        why = "độ dài 1" if len(w) == 1 else f"{w[:-1]!r} {'∈' if w[:-1] in valid else '∉'} valid"
        print(f"{w:<7} {why:<22} -> {'hợp lệ' if cond else 'loại':<7} best={best!r}")
    return best

assert trace_sort_valid(examples[1][0]) == "apple"

a       độ dài 1               -> hợp lệ  best='a'
ap      'a' ∈ valid            -> hợp lệ  best='ap'
app     'ap' ∈ valid           -> hợp lệ  best='app'
appl    'app' ∈ valid          -> hợp lệ  best='appl'
apple   'appl' ∈ valid         -> hợp lệ  best='apple'
apply   'appl' ∈ valid         -> hợp lệ  best='apple'
banana  'banan' ∉ valid        -> loại    best='apple'


| bước | từ | lý do | best |
|---|---|---|---|
| 1 | a | độ dài 1 | a |
| 2 | ap | "a" ∈ valid | ap |
| 3 | app | "ap" ∈ valid | app |
| 4 | appl | "app" ∈ valid | appl |
| 5 | apple | "appl" ∈ valid | **apple** |
| 6 | apply | "appl" ∈ valid, nhưng không dài hơn | apple |
| 7 | banana | "banan" ∉ valid | apple |

Trace Cách 4 (DFS Trie) trên **Ví dụ 3** — gốc có con `a`, `b`, `q` nhưng không con nào có cờ `end` ⇒ stack chỉ chứa gốc ⇒ trả `""`:

In [13]:
root = make_trie(examples[2][0])
print("con của gốc:", {ch: (END in node) for ch, node in root.items()})
print("kết quả:", repr(Solution().longestWord(examples[2][0])))

con của gốc: {'a': False, 'b': False, 'q': False}
kết quả: ''


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 4** — Trie + DFS chỉ qua nút kết thúc từ (đã định nghĩa ở mục 7.3, nhắc lại ở đây). Cách 2 ngắn hơn và cũng hoàn toàn đủ nhanh.

In [14]:
class Solution:
    def longestWord(self, words: List[str]) -> str:
        root = {}
        for w in words:
            node = root
            for ch in w:
                node = node.setdefault(ch, {})
            node["$"] = True
        best = ""
        stack = [(root, "")]
        while stack:
            node, path = stack.pop()
            if len(path) > len(best):
                best = path
            for ch in sorted((k for k in node if k != "$"), reverse=True):
                child = node[ch]
                if "$" in child:
                    stack.append((child, path + ch))
        return best

for words, exp in examples:
    got = Solution().longestWord(words); print(repr(got), repr(exp)); assert got == exp

'kiran' 'kiran'
'apple' 'apple'
'' ''


## 10. Vì sao thuật toán đúng

**Cách 2.** Quy nạp theo thứ tự trong `sorted(words)`. Giả sử mọi từ đứng trước `w` đã được phân loại đúng (thuộc `valid` ⇔ hợp lệ). `w[:-1]` (nếu có trong `words`) là tiền tố thật sự của `w` nên `w[:-1] < w` và đứng trước `w`. Vậy `w[:-1] ∈ valid` ⇔ `w[:-1]` hợp lệ ⇔ (cùng với `w ∈ words`) `w` hợp lệ. Trường hợp `len(w) = 1` tầm thường. Chọn `best`: duyệt tăng dần từ điển, cập nhật khi dài hơn hẳn ⇒ cuối cùng giữ từ dài nhất, và trong cùng độ dài là từ đến **sớm nhất** = nhỏ nhất. Từ trùng lặp trong `words` không ảnh hưởng.

**Cách 4.** Nút `v` được đẩy vào stack ⇔ mọi nút trên đường gốc → `v` có cờ `end` ⇔ chuỗi của `v` là từ hợp lệ. Vì thế DFS thăm **đúng** tập các từ hợp lệ. Thứ tự thăm là pre-order với con sắp tăng dần; với hai từ hợp lệ cùng độ dài `x < y`, gọi `p` là tiền tố chung dài nhất: nhánh chứa `x` (ký tự nhỏ hơn) được thăm hết trước nhánh chứa `y` ⇒ `x` được thăm trước ⇒ cập nhật nghiêm ngặt giữ `x`. ∎

### 10.1 Kiểm chứng thực nghiệm mệnh đề "truyền dọc"

Mệnh đề: `w` hợp lệ ⇔ `w ∈ words` và (`len(w) = 1` hoặc `w[:-1]` hợp lệ). Ta so sánh định nghĩa trực tiếp (mọi tiền tố) với định nghĩa đệ quy trên nhiều bộ dữ liệu ngẫu nhiên, đồng thời kiểm tra rằng tập nút được DFS của Cách 4 thăm **trùng khớp** tập từ hợp lệ.

In [15]:
def is_valid(w, S):
    return all(w[:k] in S for k in range(1, len(w) + 1))

def dfs_visited(words):
    seen, stack = set(), [(make_trie(words), "")]
    while stack:
        node, path = stack.pop()
        if path: seen.add(path)
        for ch, child in node.items():
            if ch != END and END in child:
                stack.append((child, path + ch))
    return seen

def valid_recursive(w, S, memo):
    if w not in memo:
        memo[w] = w in S and (len(w) == 1 or valid_recursive(w[:-1], S, memo))
    return memo[w]

rng = random.Random(1010)
for _ in range(400):
    ws = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(1, 14))]
    S, memo = set(ws), {}
    direct = {w for w in ws if is_valid(w, S)}
    assert direct == {w for w in ws if valid_recursive(w, S, memo)}
    assert direct == dfs_visited(ws)
print("Định nghĩa trực tiếp = đệ quy = tập DFS Trie thăm (400 test)")

Định nghĩa trực tiếp = đệ quy = tập DFS Trie thăm (400 test)


## 11. Phân tích Time / Space Complexity

Gọi `T = Σ len(words[i]) ≤ 10⁵`, `n = len(words)`.

| Cách | Time | Space |
|---|---|---|
| 1. Set + mọi tiền tố | `O(Σ mᵢ · kᵢ)` (dừng sớm), tệ nhất lớn | `O(T)` |
| 1'. Sắp `(−len, w)` + kiểm tra | như trên + sắp | `O(T)` |
| 2. Sắp + `valid` | `O(T log n)` | `O(T)` |
| 3. Trie + đi đường | `O(T)` | `O(T)` nút |
| 4. Trie + DFS nút `end` | `O(T)` | `O(T)` nút |

Đo trên dữ liệu "xấu" cho Cách 1: một chuỗi tiền tố đủ dài `1..k` và nhiều từ dài nối tiếp nó.

In [16]:
def adversarial(k, extra):
    base = "a" * k
    words = [base[:i] for i in range(1, k + 1)]
    words += ["a" * k + random.Random(i).choice("bcd") * 1 for i in range(extra)]
    return words

data = adversarial(300, 150)
print("tổng độ dài T =", sum(map(len, data)))
for name, f in [("brute", longest_bruteforce), ("sort_valid", longest_sort_valid), ("trie_walk", longest_trie_walk), ("submit", Solution().longestWord)]:
    t = time.perf_counter(); r = f(data); print(f"{name:<11} -> len {len(r)} ({(time.perf_counter()-t)*1000:.1f} ms)")

tổng độ dài T = 90300
brute       -> len 301 (7.8 ms)
sort_valid  -> len 301 (0.1 ms)
trie_walk   -> len 301 (5.0 ms)
submit      -> len 301 (2.1 ms)


## 12. Bộ test

In [17]:
ALL = {"brute": longest_bruteforce, "sorted_first": longest_sorted_first, "sort_valid": longest_sort_valid,
       "trie_walk": longest_trie_walk, "submit": Solution().longestWord}
def run_all(words):
    res = {k: f(list(words)) for k, f in ALL.items()}
    assert len(set(res.values())) == 1, (words, res)
    return res["submit"]

for words, exp in examples:
    assert run_all(words) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — đã kiểm bằng vét cạn trong run_all
edge = [(["b", "a"], "a"), (["abc"], ""), (["a"], "a"), (["a", "a", "ab"], "ab"),
        (["a", "ab", "abc", "b", "bc", "bcd"], "abc"), (["a", "abc", "abcd"], "a"),
        (["z", "zy", "y", "yz"], "yz")]
for words, exp in edge:
    assert run_all(words) == exp, (words, run_all(words))
rng = random.Random(1858)
for _ in range(800):
    alpha = "ab" if rng.random() < 0.5 else "abc"
    words = ["".join(rng.choice(alpha) for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 15))]
    run_all(words)
long_word = ["x" * i for i in range(1, 300)]
assert run_all(long_word) == "x" * 299
print("Ví dụ, 7 edge case, 800 test ngẫu nhiên, chuỗi tiền tố dài 299: OK")

Ví dụ, 7 edge case, 800 test ngẫu nhiên, chuỗi tiền tố dài 299: OK


### 12.1 Kiểm tra tính chất

- Hoán vị `words` không đổi đáp án.
- Thêm bản sao trùng không đổi đáp án.
- Đáp án (nếu khác rỗng) có mọi tiền tố trong `words`, và không có từ hợp lệ nào dài hơn hoặc cùng độ dài mà nhỏ hơn.

In [18]:
rng = random.Random(121)
f = Solution().longestWord
for _ in range(400):
    words = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 12))]
    ans = f(words)
    sh = words[:]; rng.shuffle(sh)
    assert f(sh) == ans and f(words + words[:3]) == ans
    S = set(words)
    if ans:
        assert is_valid(ans, S)
    assert not any(is_valid(w, S) and (len(w) > len(ans) or (len(w) == len(ans) and w < ans)) for w in words)
print("3 tính chất đúng trên 400 test")

3 tính chất đúng trên 400 test


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra `w[:-1] in words` (không truyền dọc)

In [19]:
def wrong_only_parent(words):
    S, best = set(words), ""
    for w in sorted(words):
        if len(w) == 1 or w[:-1] in S:                 # SAI: S là tập mọi từ, không phải tập từ hợp lệ
            if len(w) > len(best):
                best = w
    return best

words = ["a", "abc", "abcd"]       # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", repr(wrong_only_parent(words)), "| đúng:", repr(Solution().longestWord(words)), "| brute:", repr(longest_bruteforce(words)))

sai : 'abcd' | đúng: 'a' | brute: 'a'


**Vì sao sai:** `"abcd"` có `"abc"` trong danh sách nhưng `"ab"` thì không ⇒ `"abcd"` không hợp lệ. Kiểm tra cha trực tiếp chỉ đúng khi tập kiểm tra là **tập từ hợp lệ** (Cách 2), không phải tập mọi từ. **Sửa:** `valid` set, thêm `w` vào chỉ khi `w` hợp lệ.

### Lỗi 2 — Tie-break sai (cập nhật khi `>=`)

In [20]:
def wrong_tie(words):
    valid, best = set(), ""
    for w in sorted(words):
        if len(w) == 1 or w[:-1] in valid:
            valid.add(w)
            if len(w) >= len(best):                    # SAI: từ sau (lớn hơn) đè lên
                best = w
    return best

words, exp = examples[1]
print("sai :", repr(wrong_tie(words)), "| đúng:", repr(exp))

sai : 'apply' | đúng: 'apple'


**Vì sao sai:** duyệt tăng dần từ điển nên từ cùng độ dài đến **sau** lại lớn hơn; `>=` khiến `"apply"` đè `"apple"`. **Sửa:** `>` nghiêm ngặt (hoặc so sánh tường minh `w < best` khi cùng độ dài).

### Lỗi 3 — DFS Trie tìm đường dài nhất mà không đòi cờ `end`

In [21]:
def wrong_dfs_no_end(words):
    root, best = make_trie(words), ""
    stack = [(root, "")]
    while stack:
        node, path = stack.pop()
        if len(path) > len(best):
            best = path
        for ch in sorted((k for k in node if k != END), reverse=True):
            stack.append((node[ch], path + ch))          # SAI: không kiểm tra END
    return best

words, exp = examples[2]
print("sai :", repr(wrong_dfs_no_end(words)), "| đúng:", repr(exp))

sai : 'abc' | đúng: ''


**Vì sao sai:** đường dài nhất trong Trie chỉ là "từ dài nhất", không quan tâm các tiền tố có là từ hay không. **Sửa:** chỉ đi vào con có cờ `end`.

### Lỗi 4 — DFS duyệt con theo thứ tự tùy ý (thứ tự chèn của dict)

In [22]:
def wrong_dfs_order(words):
    root, best = make_trie(words), ""
    stack = [(root, "")]
    while stack:
        node, path = stack.pop()
        if len(path) > len(best):
            best = path
        for ch in [k for k in node if k != END]:          # SAI: không sắp, lại còn pop ngược
            if END in node[ch]:
                stack.append((node[ch], path + ch))
    return best

words = ["a", "b", "ba", "ab"]     # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", repr(wrong_dfs_order(words)), "| đúng:", repr(Solution().longestWord(words)), "| brute:", repr(longest_bruteforce(words)))

sai : 'ba' | đúng: 'ab' | brute: 'ab'


**Vì sao sai:** dict giữ thứ tự chèn (`a` rồi `b`), stack lấy phần tử cuối (`b`) ra trước nên `"ba"` được thấy trước `"ab"` ⇒ thứ tự thăm phụ thuộc dữ liệu vào, không phải từ điển. **Sửa:** đẩy con theo thứ tự **giảm dần** để lấy ra tăng dần, hoặc so sánh tường minh khi cùng độ dài.

### Lỗi 5 — Dựng tập mọi tiền tố bằng cắt chuỗi cho từ rất dài (không dừng sớm)

In [23]:
def slow_all_prefixes(words):
    S, best = set(words), ""
    for w in words:
        prefs = {w[:k] for k in range(1, len(w) + 1)}    # O(m²) ký tự kể cả khi thiếu ngay tiền tố đầu
        if prefs <= S and (len(w) > len(best) or (len(w) == len(best) and w < best)):
            best = w
    return best

data = ["q" * 15000, "a", "ab"]
for f in (slow_all_prefixes, Solution().longestWord):
    t = time.perf_counter(); r = f(data); print(f"{f.__name__:<18} -> {r!r} ({(time.perf_counter()-t)*1000:.0f} ms)")

slow_all_prefixes  -> 'ab' (72 ms)
longestWord        -> 'ab' (2 ms)


**Vì sao chậm:** dựng mọi tiền tố của từ dài `m` tốn `Θ(m²)` ký tự, kể cả khi tiền tố `"q"` đã thiếu ngay từ đầu. Với `m = 10⁵` là `5·10⁹`. **Sửa:** dừng sớm (`all(...)` lười), hoặc Trie/Cách 2.

### Lỗi 6 — Quên trường hợp không có đáp án / trả `None`

In [24]:
def wrong_none(words):
    S = set(words)
    cands = [w for w in words if all(w[:k] in S for k in range(1, len(w)))]
    return min(cands, key=lambda x: (-len(x), x)) if cands else None     # SAI: phải trả ""

print("sai :", repr(wrong_none(examples[2][0])), "| đúng:", repr(examples[2][1]))

sai : None | đúng: ''


**Vì sao sai:** đề yêu cầu chuỗi rỗng `""` khi không có từ thỏa. **Sửa:** khởi tạo `best = ""`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Mọi tiền tố** đều có mặt" ⇒ tính chất truyền dọc: chỉ cần kiểm tra **cha trực tiếp** trên tập **đã hợp lệ**.
- `sorted(strings)` đặt tiền tố trước từ chứa nó — dùng được như thứ tự topo của quan hệ "là tiền tố".
- Trie: mỗi đường = một tiền tố; cờ `end` = "là một từ". "Mọi tiền tố là từ" = "mọi nút trên đường có `end`".
- Tie-break theo từ điển: duyệt theo thứ tự tăng dần + cập nhật nghiêm ngặt; hoặc sắp theo khóa `(−len, w)`.
- Bài gần như giống hệt **720 Longest Word in Dictionary** (không Premium) — luyện ở đó nếu không có Premium.

## 15. Biến thể của bài

1. **720 Longest Word in Dictionary**: cùng yêu cầu, ràng buộc nhỏ hơn.
2. **Đếm số từ hợp lệ** thay vì tìm dài nhất — Cách 2 chỉ cần `len(valid)`.
3. **2416 Sum of Prefix Scores of Strings**: với mỗi từ, cộng số từ có tiền tố là từng tiền tố của nó — Trie đếm (Bài tập 5).
4. **Cho phép thiếu tối đa một tiền tố** — DFS Trie với tham số "đã dùng quyền bỏ qua".
5. **1065 / 211**: Trie với ký tự đại diện, tìm kiếm tiền tố.

### 15.1 Code biến thể: liệt kê mọi từ hợp lệ theo thứ tự từ điển (DFS Trie)

In [25]:
def all_valid_words(words):
    root, out = make_trie(words), []
    stack = [(root, "")]
    while stack:
        node, path = stack.pop()
        if path:
            out.append(path)
        for ch in sorted((k for k in node if k != END), reverse=True):
            if END in node[ch]:
                stack.append((node[ch], path + ch))
    return out

w = examples[1][0]
print(all_valid_words(w))
assert all_valid_words(w) == sorted({x for x in w if is_valid(x, set(w))})

['a', 'ap', 'app', 'appl', 'apple', 'apply']


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Set + mọi tiền tố | Kiểm tra trực tiếp | lớn tệ nhất | `O(T)` | Kiểm chứng, dữ liệu nhỏ |
| 1'. Sắp `(−len, w)` | Trả từ hợp lệ đầu tiên | như 1 | `O(T)` | Tie-break gọn |
| 2. Sắp + `valid` | Truyền dọc qua cha | `O(T log n)` | `O(T)` | Ngắn, dễ nhớ |
| 3. Trie + đi đường | Cờ `end` mọi nút | `O(T)` | `O(T)` | Khi đã có Trie |
| 4. Trie + DFS | Chỉ đi nút `end` | `O(T)` | `O(T)` | **Bản nộp**, mở rộng tốt |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `["a","b","c"]`; (b) `["ab","a","abc","b"]`; (c) `["z"]`; (d) `["xy","x","xyz","w","wx"]`.

**Bài 2 (tính tay):** Trace Cách 2 trên `["t","to","tea","te","ten","i","in","inn"]` — liệt kê `valid` sau mỗi bước và `best` cuối cùng.

**Bài 3 (điền TODO):** Hoàn thiện `is_valid_todo(w, S)` dùng dừng sớm.

**Bài 4 (tự viết lại):** Viết `count_valid(words)` trả số từ **khác nhau** thỏa điều kiện, dùng Trie.

**Bài 5 (mini challenge):** Viết `sum_prefix_scores(words)` (LeetCode 2416): với mỗi `words[i]`, tổng qua mọi tiền tố `p` của nó của "số từ trong `words` có `p` là tiền tố". Dùng Trie đếm, `O(T)`; đối chiếu vét cạn.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def is_valid_todo(w, S):
    # TODO: duyệt k = 1..len(w); nếu w[:k] không có trong S thì trả False ngay
    pass

In [27]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def count_valid(words):
    # TODO: dựng Trie; DFS chỉ qua nút END; đếm số nút được thăm (trừ gốc)
    pass

def sum_prefix_scores(words):
    # TODO: Trie với bộ đếm cnt ở mỗi nút (cộng 1 khi đi qua lúc chèn); với mỗi từ, cộng cnt dọc đường
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tìm từ dài nhất mà mọi tiền tố có mặt. Mức 2: cùng độ dài ⇒ nhỏ nhất. Mức 3: (a) `"a"`; (b) `"abc"`; (c) `"z"`; (d) `"xyz"`.

**Bài 2.** Mức 1: sắp: `i, in, inn, t, te, tea, ten, to`. Mức 2: mọi từ đều hợp lệ. Mức 3: dài nhất 3 là `inn`, `tea`, `ten` ⇒ `"inn"`.

**Bài 3.** Mức 1: vòng `for`. Mức 2: `return False` khi thiếu. Mức 3: `return True` cuối vòng.

**Bài 4.** Mức 1: từ trùng chỉ tạo một nút. Mức 2: DFS như Cách 4. Mức 3: đếm số lần pop trừ gốc.

**Bài 5.** Mức 1: `cnt[node]` = số từ đi qua `node`. Mức 2: điểm của từ = tổng `cnt` các nút trên đường. Mức 3: trả list cùng thứ tự `words`.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1, 2, 3
f = Solution().longestWord
b1 = [f(w) for w in [["a", "b", "c"], ["ab", "a", "abc", "b"], ["z"], ["xy", "x", "xyz", "w", "wx"]]]
print("Bài 1:", b1); assert b1 == ["a", "abc", "z", "xyz"]
b2 = ["t", "to", "tea", "te", "ten", "i", "in", "inn"]
print("Bài 2:"); assert trace_sort_valid(b2) == "inn"
def is_valid_todo(w, S):
    for k in range(1, len(w) + 1):
        if w[:k] not in S:
            return False
    return True
assert is_valid_todo("apple", set(examples[1][0])) and not is_valid_todo("banana", set(examples[1][0]))
print("Bài 3: OK")

Bài 1: ['a', 'abc', 'z', 'xyz']
Bài 2:
i       độ dài 1               -> hợp lệ  best='i'
in      'i' ∈ valid            -> hợp lệ  best='in'
inn     'in' ∈ valid           -> hợp lệ  best='inn'
t       độ dài 1               -> hợp lệ  best='inn'
te      't' ∈ valid            -> hợp lệ  best='inn'
tea     'te' ∈ valid           -> hợp lệ  best='inn'
ten     'te' ∈ valid           -> hợp lệ  best='inn'
to      't' ∈ valid            -> hợp lệ  best='inn'
Bài 3: OK


In [29]:
# Bài 4, 5
def count_valid(words):
    root, cnt = make_trie(words), 0
    stack = [root]
    while stack:
        node = stack.pop()
        for ch, child in node.items():
            if ch != END and END in child:
                cnt += 1; stack.append(child)
    return cnt

def sum_prefix_scores(words):
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {"#": 0}); node["#"] += 1
    res = []
    for w in words:
        node, s = root, 0
        for ch in w:
            node = node[ch]; s += node["#"]
        res.append(s)
    return res

rng = random.Random(5)
for _ in range(300):
    ws = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 10))]
    assert count_valid(ws) == len({w for w in ws if is_valid(w, set(ws))})
    assert sum_prefix_scores(ws) == [sum(sum(x.startswith(w[:k]) for x in ws) for k in range(1, len(w) + 1)) for w in ws]
print("Bài 4, 5: OK |", count_valid(examples[1][0]), sum_prefix_scores(["abc", "ab", "bc", "b"]), "(2416: [5,4,3,2])")

Bài 4, 5: OK | 6 [5, 4, 3, 2] (2416: [5,4,3,2])


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (sắp xếp + tập hợp lệ):**

```python
valid, best = set(), ""
for w in sorted(words):                 # tiền tố đứng trước
    if len(w) == 1 or w[:-1] in valid:
        valid.add(w)
        if len(w) > len(best):          # '>' giữ từ nhỏ hơn theo từ điển
            best = w
return best
```

**Template (Trie DFS):** chèn mọi từ, đánh `end`; DFS từ gốc chỉ vào con có `end`, đẩy con theo thứ tự giảm dần để lấy ra tăng dần; cập nhật khi sâu hơn hẳn.

**Checklist:**
- [ ] Kiểm tra cha trên tập **hợp lệ**, không phải tập mọi từ?
- [ ] Tie-break: cùng độ dài lấy từ **nhỏ hơn**?
- [ ] Trả `""` khi không có?
- [ ] Từ trùng lặp không làm sai?
- [ ] Không dựng `Θ(m²)` tiền tố cho từ dài?